# Cardiac JEPA — pré-entraînement sur Colab

Entraîne le JEPA latent sur PTB-XL (100 Hz) avec le GPU Colab.

**Avant de lancer** : `Exécution ▸ Modifier le type d'exécution ▸ GPU`.

Ce notebook est *idempotent* : re-lance-le après une déconnexion, il reprend
l'entraînement là où il s'était arrêté (`--resume auto`).

**Une seule chose à renseigner** : `REPO_URL` dans la cellule de configuration.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print("torch", torch.__version__, "| cuda dispo :", torch.cuda.is_available())

## 2. Monter Drive et configurer les chemins

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# >>> À RENSEIGNER <<<
REPO_URL = "https://github.com/<ton-user>/cardiac-jepa.git"

DRIVE_ROOT  = Path('/content/drive/MyDrive/cardiac-jepa')   # persiste entre sessions
DATA_DRIVE  = DRIVE_ROOT / 'data'                            # les 2 CSV (petits)
CACHE_DRIVE = DRIVE_ROOT / 'ptbxl_100hz.npy'                 # cache signaux (1,05 Go)
CACHE_LOCAL = Path('/content/ptbxl_100hz.npy')               # copie sur disque local = rapide
RUN_DIR     = DRIVE_ROOT / 'runs' / 'tiny_v1'                # checkpoints + metrics.csv
REPO_DIR    = Path('/content/cardiac-jepa')

for d in (DATA_DRIVE, RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Drive prêt :", DRIVE_ROOT)

## 3. Cloner / mettre à jour le code

Tu édites en local dans VSCode, tu `git push`, puis tu ré-exécutes cette cellule.

In [ ]:
if REPO_DIR.exists():
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -r requirements.txt

## 4. Données

**Première session** : télécharge le zip PhysioNet (1,84 Go), construit le cache `.npy`,
et le sauve sur Drive. Compte ~10 min.

**Sessions suivantes** : copie simplement le `.npy` depuis Drive (~1 min). Les 21 837
fichiers `.dat` ne sont alors plus nécessaires.

Pourquoi un cache : lire 21 837 petits fichiers depuis un Drive monté est *atrocement* lent.
Un seul fichier contigu sur le disque local de Colab, c'est instantané.

In [ ]:
import shutil, time

ZIP_URL = "https://physionet.org/content/ptb-xl/get-zip/1.0.1/"
EXTRACT = Path('/content/ptbxl_raw')
DS_NAME = "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"

if not CACHE_DRIVE.exists():
    print("Cache absent sur Drive -> téléchargement + construction (une seule fois).")
    t0 = time.time()
    !wget -q --show-progress -O /content/ptbxl.zip "{ZIP_URL}"
    !mkdir -p {EXTRACT} && unzip -q -o /content/ptbxl.zip -d {EXTRACT}
    raw = EXTRACT / DS_NAME
    assert (raw / 'ptbxl_database.csv').exists(), f"arborescence inattendue: {list(EXTRACT.iterdir())}"

    !python -m jepa.build_cache --data-dir "{raw}" --out "{CACHE_LOCAL}"

    print("Copie du cache + CSV vers Drive...")
    shutil.copy(CACHE_LOCAL, CACHE_DRIVE)
    for csv in ('ptbxl_database.csv', 'scp_statements.csv'):
        shutil.copy(raw / csv, DATA_DRIVE / csv)
    print(f"Fait en {time.time()-t0:.0f}s")
else:
    if not CACHE_LOCAL.exists():
        print("Copie du cache Drive -> disque local...")
        shutil.copy(CACHE_DRIVE, CACHE_LOCAL)
    print("Cache prêt :", CACHE_LOCAL)

# Le code lit ces variables d'environnement (voir jepa/data.py).
os.environ['PTBXL_DATA_DIR'] = str(DATA_DRIVE)   # CSV uniquement
os.environ['PTBXL_CACHE']    = str(CACHE_LOCAL)

## 5. Sanity check : les données se chargent

In [ ]:
from jepa.data import PTBXLDataset
ds = PTBXLDataset('pretrain')
x = ds[0]
print(f"pretrain={len(ds)} ECG | un échantillon: {tuple(x.shape)} "
      f"| moyenne={x.mean():.3f} std={x.std():.3f}  (z-normé par dérivation)")

## 6. Entraînement

`--resume auto` reprend depuis `latest.pt` s'il existe. Si Colab te déconnecte,
ré-exécute simplement cette cellule : tu perds au plus une epoch.

In [ ]:
!python -m jepa.train \
    --config jepa/configs/tiny.yaml \
    --out "{RUN_DIR}" \
    --resume auto \
    --workers 2

## 7. Courbes de collapse

Le critère de succès de cette phase. À surveiller :
- `emb_std_ctx` **ne doit pas tendre vers 0**,
- `eff_rank_ctx` doit rester élevé,
- `jepa` doit baisser *progressivement* (pas s'effondrer à ~0 d'emblée),
- `eff_rank_tgt` : point de vigilance connu — la VICReg ne régularise que le contexte.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

m = pd.read_csv(RUN_DIR / 'metrics.csv')
tr, va = m[m.phase == 'train'], m[m.phase == 'val']

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(tr.step, tr.jepa); ax[0].set_title('L_jepa (train)'); ax[0].set_xlabel('step')
ax[1].plot(va.epoch, va.emb_std_ctx, label='contexte')
ax[1].plot(va.epoch, va.emb_std_tgt, label='cible')
ax[1].plot(va.epoch, va.pred_std, label='predictor')
ax[1].axhline(0.1, ls='--', c='r', lw=1, label='seuil collapse')
ax[1].set_title('écart-type des embeddings'); ax[1].set_xlabel('epoch'); ax[1].legend()
ax[2].plot(va.epoch, va.eff_rank_ctx, label='contexte')
ax[2].plot(va.epoch, va.eff_rank_tgt, label='cible')
ax[2].set_title('rang effectif (max 192)'); ax[2].set_xlabel('epoch'); ax[2].legend()
plt.tight_layout(); plt.show()